In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

<font size="5" color="black">ch07_1.다중 클래스 분류, 콜백 함수(mnist)</font>

- MNIST : 0부터 9까지의 손글씨 숫자 이미지로 구성된 머신러닝 및 딥러닝 분야의 대표적인 표준 벤치마크 데이터셋

In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras.datasets import mnist # mnist dataset
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from tensorflow.keras.models import Sequential, save_model, load_model
from tensorflow.keras.layers import Dense, Input, Dropout, BatchNormalization
from tensorflow.keras.layers import ReLU, ELU, LeakyReLU
from tensorflow.keras.regularizers import l2
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.optimizers import SGD
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# 0. 데이터셋 불러오기

## 1. 파일을 통해 불러오기

In [ ]:
data = np.loadtxt('data/mnist_train_small.csv', delimiter=',', skiprows=1)
data.shape

In [ ]:
print('타겟변수 : ', data[0, 0])
image = data[50, 1:].reshape(28,28)

In [ ]:
for row in image:
    for pix in row:
        print(f"{pix:3.0f}", end='')
    print()

In [ ]:
plt.rcParams['figure.figsize']=(2,2)
plt.imshow(image, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
# 컬러 사진 >> 흑백 모드로 데이터 읽기 (넘파이 배열)

In [ ]:
%pip install opencv-python==4.8.1.78

In [ ]:
import cv2
# 사진 이미지를 넘파이 배열로 읽어오기
image_array = cv2.imread('data/sample.jpg', cv2.IMREAD_GRAYSCALE) # 흑백 포맷
image_array.shape, type(image_array)

In [ ]:
# 넘파이 배열 이미지화
plt.figure(figsize=(3,4))
plt.imshow(image_array, cmap='gray') # 컬러맵 적용
plt.axis('off')
plt.show()

## 2. 데이터셋을 통해 가져오기

In [ ]:
(X_train, y_train),(X_test, y_test) = mnist.load_data() # 6만, 1만개의 학습, 시험 데이터셋으로 분리되어 로드됨

# 1. 데이터 전처리

In [ ]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

In [ ]:
value, counts = np.unique(y_train, return_counts=True)
ratios = counts/y_train.shape[0]
dict(zip(value, ratios))

In [ ]:
print(y_test[1])
plt.imshow(X_test[1], cmap='gray')
plt.show()
for row in X_test[1]:
    for pix in row:
        print(f"{pix:3.0f}", end='')
    print()

In [ ]:
X_val = X_train[50000:]
y_val = y_train[50000:]
X_train = X_train[:50000]
y_train = y_train[:50000]

In [ ]:
print('학습 데이터셋 :', X_train.shape, y_train.shape)
print('검증 데이터셋 :', X_val.shape, y_val.shape)
print('시험 데이터셋 :', X_test.shape, y_test.shape)

In [ ]:
# 입력 변수 (n, 28, 28) >> (n, 28*28) >> 스케일 조정(/255.0)
train_X = X_train.reshape(50000, 28*28).astype('float32')/255.0
val_X = X_val.reshape(10000, 784).astype('float32')/255.0
test_X = X_test.reshape(10000, -1).astype('float32')/255.0
train_X.shape, val_X.shape, test_X.shape

In [ ]:
# 종속변수 : 분류분석을 위한 원-핫 인코딩 (to_categorical(), pd.getdummies() 등 활용)
train_Y = to_categorical(y_train, 10)
val_Y = to_categorical(y_val)
test_Y = to_categorical(y_test)
train_Y.shape, val_Y.shape, test_Y.shape

In [ ]:
# train : 임의의 700개, val : 임의의 300개 추출 (수업 중 시간 절약)
train_idx = np.random.choice(50000, 700, replace=False) # 비복원추출(중복x)
val_idx = np.random.choice(10000, 300, replace=False)
train_X = train_X[train_idx]
train_Y = train_Y[train_idx]
val_X = val_X[val_idx]
val_Y = val_Y[val_idx]

# 2. 모델 구성 & 학습과정 설정 & 학습

In [ ]:
model = Sequential(name="mnist_seq")
model.add(Input(shape=(784,)))

model.add(Dense(units=128, activation='elu', kernel_regularizer=l2(0.01)))
model.add(BatchNormalization())
model.add(Dropout(0.3))

model.add(Dense(units=64, activation='elu', kernel_regularizer=l2(0.01)))
model.add(BatchNormalization())
model.add(Dropout(0.3))

model.add(Dense(units=10, activation='softmax'))

model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

hist = model.fit(train_X, train_Y,
                epochs=200,
                batch_size=64,
                validation_data=(val_X, val_Y),
                verbose=1)

# 3. 모델 시각화 & 평가

In [ ]:
fig, loss_ax = plt.subplots(figsize=(10,6))
loss_ax.plot(hist.history['loss'], 'y', label='train_loss')
loss_ax.plot(hist.history['val_loss'], 'r', label='val_loss')
loss_ax.set_xlabel('epoch')
loss_ax.set_ylabel('loss', rotation=0)

acc_ax = loss_ax.twinx()
acc_ax.plot(hist.history['accuracy'], 'g', label='train_accuracy')
acc_ax.plot(hist.history['val_accuracy'], 'b', label='val_accuracy')
acc_ax.set_ylabel('accuracy')
loss_ax.legend(loc='center right', bbox_to_anchor=(1.5, 0.6), ncol=1)
acc_ax.legend(loc='center right', bbox_to_anchor=(1.5, 0.8), ncol=1)
plt.show()

In [ ]:
loss, accuracy = model.evaluate(test_X, test_Y)
print(f'loss : {loss:.2f}')
print(f'정확도 : {accuracy*100:.2f}%')

# accuracy control
- 위 모델(DNN)의 accuracy 높이기
1. 데이터 추가 확보
2. 모델 수정 (레이어 추가, units 증가)
 - 과적합 방지(validation data, 활성화 함수를 relu 계열, tahn, Dropout, 배치정규화, L2 규제 등)
3. epoch 및 optimizer 변경

In [ ]:
%%time
# 데이터 생성
(X_train, y_train),(X_test, y_test) = mnist.load_data()

X_val = X_train[50000:]
y_val = y_train[50000:]
X_train = X_train[:50000]
y_train = y_train[:50000]

# 데이터 전처리
train_X = X_train.reshape(50000, 28*28).astype('float32')/255.0
val_X = X_val.reshape(10000, 784).astype('float32')/255.0
test_X = X_test.reshape(10000, -1).astype('float32')/255.0

train_Y = to_categorical(y_train, 10)
val_Y = to_categorical(y_val)
test_Y = to_categorical(y_test)

# # 수업용 데이터 사이즈 조정
# train_idx = np.random.choice(50000, 20000, replace=False)
# val_idx = np.random.choice(10000, 5000, replace=False)
# train_X = train_X[train_idx]
# train_Y = train_Y[train_idx]
# y_train_sub = y_train[train_idx]

val_X = val_X[val_idx]
val_Y = val_Y[val_idx]

# 모델 생성
model = Sequential(name="mnist_seq")
model.add(Input(shape=(784,)))

model.add(Dense(units=256, activation='elu', kernel_regularizer=l2(0.0002)))
model.add(BatchNormalization())
model.add(Dropout(0.4))

model.add(Dense(units=128, activation='gelu', kernel_regularizer=l2(0.0002)))
model.add(BatchNormalization())
model.add(Dropout(0.3))

model.add(Dense(units=64, activation='elu', kernel_regularizer=l2(0.0002)))
model.add(BatchNormalization())
model.add(Dropout(0.2))

model.add(Dense(units=10, activation='softmax'))

# 학습 과정 설정
sgd_optimizer = SGD(learning_rate=0.01, momentum=0.9, nesterov=True)

model.compile(loss='categorical_crossentropy',
              optimizer=sgd_optimizer,
              metrics=['accuracy'])

# 클래스 불균형 극복
unique_classes = np.unique(y_train_sub)

class_weight = compute_class_weight(
    class_weight='balanced',
    classes=unique_classes,
    y=y_train_sub
)

class_weight_dict = dict(enumerate(class_weight))

# 콜백 설정
lr_reduce = [
ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=0),
EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0)
            ]

# 학습
hist = model.fit(train_X, train_Y,
                epochs=200,
                batch_size=512,
                validation_data=(val_X, val_Y),
                class_weight=class_weight_dict,
                callbacks=[lr_reduce],
                verbose=0)

# 시각화
fig, loss_ax = plt.subplots(figsize=(10,6))
loss_ax.plot(hist.history['loss'], 'y', label='train_loss')
loss_ax.plot(hist.history['val_loss'], 'r', label='val_loss')
loss_ax.set_xlabel('epoch')
loss_ax.set_ylabel('loss', rotation=0)

acc_ax = loss_ax.twinx()
acc_ax.plot(hist.history['accuracy'], 'g', label='train_accuracy')
acc_ax.plot(hist.history['val_accuracy'], 'b', label='val_accuracy')
acc_ax.set_ylabel('accuracy')
loss_ax.legend(loc='center right', bbox_to_anchor=(1.5, 0.6), ncol=1)
acc_ax.legend(loc='center right', bbox_to_anchor=(1.5, 0.8), ncol=1)
plt.show()

# 평가
loss, accuracy = model.evaluate(test_X, test_Y)
print(f'loss : {loss:.2f}')
print(f'accuracy : {accuracy*100:.2f}%')

In [ ]:
# 교차표
y_test
test_Y.argmax(axis=1)
np.all(test_Y.argmax(axis=1) == y_test)

In [ ]:
y_hat = model.predict(test_X).argmax(axis=1)
y_hat

In [ ]:
pd.crosstab(y_test, y_hat, rownames=['실제값'], colnames=['예측값'])

In [ ]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, y_hat)

In [ ]:
# 예측이 틀린 개수
10000 - accuracy * 10000

# 콜백 함수

In [ ]:
from tensorflow.keras.callbacks import Callback, EarlyStopping, ModelCheckpoint
from tensorflow.keras.callbacks import ReduceLROnPlateau

## 1. 로그 출력 통제

In [ ]:
# 특정 에포크마다 출력하도록 로그 출력을 통제하는 Callback 클래스
class CustomHistory(Callback):
    def __init__(self):
        self.epoch = 0
    
    def on_epoch_end(self, batch, logs={}):
        '1 epoch 수행 후 자동 실행되는 함수'
        self.epoch += 1
        if self.epoch%10==0:
            print('epoch:{}, loss:{:.4f}, acc:{:.4f}, val_loss:{:.4f}, val_acc:{:.4f}'\
                  .format(self.epoch,
                         logs.get('loss', '-'),
                         logs.get('accuracy', '-'),
                         logs.get('val_loss', '-'),
                         logs.get('val_accuracy', '-')
                         )
                 )

## 2. 조기 종료
- EarlyStopping : 검증 성능이 더 이상 개선되지 않을 때 학습을 중간에 멈추는 규제 기법

In [ ]:
earlyStopping = EarlyStopping(monitor='val_accuracy',
                              patience=30,
                              verbose=0)

## 3. 최적 모델 저장
- ModelCheckpoint : 특정 시점의 모델 가중치 또는 전체 구조를 디스크에 주기적으로 저장

In [ ]:
import os
model_save_folder = 'model07/'
if not os.path.exists(model_save_folder):
    os.mkdir(model_save_folder)
    
file = model_save_folder + 'mnist_{epoch:03d}_val_{val_accuracy:.4f}.h5'
    
modelCheckpoint = ModelCheckpoint(filepath=file, # 파일 저장 경로
                                  monitor='val_accuracy', # 기준 지표
                                  verbose=0,
                                  save_best_only=True, # 기준 지표가 개선된 경우 저장
                                  mode = 'max' # 이전 최댓값보다 커질 경우 저장 (monitor='val_accuracy'로 설정되어 있으면 Keras 내부적으로 알아서 mode='max'를 인식)
                                 )

## 4. 학습률 조정
- ReduceLROnPlateau : 모델의 성능 향상이 정체되었을 때 학습률(LR)을 자동으로 줄여주는 기법

In [ ]:
reduce_lr = ReduceLROnPlateau(monitor='val_accuracy', 0# 기준 지표 (기본값 : val_loss)
                              patience=5, # 정체 판단 epoch 기준
                              factor=0.5, # 정체 시 학습률을 줄일 비율
                              min_lr=1e-6 # 학습률의 최저 하한선
                             )

In [ ]:
%%time

# 데이터 생성
(X_train, y_train),(X_test, y_test) = mnist.load_data()

X_val = X_train[50000:]
y_val = y_train[50000:]
X_train = X_train[:50000]
y_train = y_train[:50000]

# 데이터 전처리
train_X = X_train.reshape(50000, 28*28).astype('float32')/255.0
val_X = X_val.reshape(10000, 784).astype('float32')/255.0
test_X = X_test.reshape(10000, -1).astype('float32')/255.0

train_Y = to_categorical(y_train, 10)
val_Y = to_categorical(y_val)
test_Y = to_categorical(y_test)

# 수업용 데이터 사이즈 조정
train_idx = np.random.choice(50000, 1000, replace=False)
val_idx = np.random.choice(10000, 300, replace=False)
train_X = train_X[train_idx]
train_Y = train_Y[train_idx]
y_train_sub = y_train[train_idx]

val_X = val_X[val_idx]
val_Y = val_Y[val_idx]

# 모델 생성
model = Sequential(name="mnist_seq")
model.add(Input(shape=(784,)))

model.add(Dense(units=256, activation='elu', kernel_regularizer=l2(0.0002)))
model.add(BatchNormalization())
model.add(Dropout(0.4))

model.add(Dense(units=64, activation='elu', kernel_regularizer=l2(0.0002)))
model.add(BatchNormalization())
model.add(Dropout(0.2))

model.add(Dense(units=10, activation='softmax'))

# 학습 과정 설정
model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

# 학습
customHistory = CustomHistory()
print(customHistory.epoch)

hist = model.fit(train_X, train_Y,
                epochs=200,
                batch_size=64,
                validation_data=(val_X, val_Y),
                verbose=0,
                callbacks=[customHistory, earlyStopping, modelCheckpoint, reduce_lr]
                )

# 시각화
fig, loss_ax = plt.subplots(figsize=(10,6))
loss_ax.plot(hist.history['loss'], 'y', label='train_loss')
loss_ax.plot(hist.history['val_loss'], 'r', label='val_loss')
loss_ax.set_xlabel('epoch')
loss_ax.set_ylabel('loss', rotation=0)

acc_ax = loss_ax.twinx()
acc_ax.plot(hist.history['accuracy'], 'g', label='train_accuracy')
acc_ax.plot(hist.history['val_accuracy'], 'b', label='val_accuracy')
acc_ax.set_ylabel('accuracy')
loss_ax.legend(loc='center right', bbox_to_anchor=(1.5, 0.6), ncol=1)
acc_ax.legend(loc='center right', bbox_to_anchor=(1.5, 0.8), ncol=1)
plt.show()

# 평가
loss, accuracy = model.evaluate(test_X, test_Y)
print(f'loss : {loss:.2f}')
print(f'accuracy : {accuracy*100:.2f}%')

In [ ]:
# 최종 학습 모델
model.evaluate(test_X, test_Y)

In [ ]:
model2 = load_model('model07/mnist_046_val_0.9200.h5')

In [ ]:
# 체크포인트 모델
model2.evaluate(test_X, test_Y)